# SetBased Model on Cranfield Collection

This notebook demonstrates the SetBased information retrieval model on the Cranfield (CF) collection.

**Model:** Pure set-based retrieval with Apriori termset mining and IDF weighting, no graph-based node weighting.

**Collection:** CF has 1,239 documents and 100 queries with relevance judgments.

## Setup

In [ ]:
from pathlib import Path
from graph_ir.data.collection import Collection
from graph_ir.models.setbased import SetBasedModel
import numpy as np

# Find collection
cf_path = Path("collections/cf")
print(f"Loading CF from {cf_path}...")

## Load Collection

In [ ]:
# Load collection (load first 10 queries for a quick demo)
collection = Collection(cf_path).create()
collection.load_queries().load_qrels()

print(f"\nCollection stats:")
print(f"  Documents: {collection.num_docs}")
print(f"  Queries: {len(collection.queries)}")
print(f"  Vocabulary: {len(collection.inverted_index)} terms")
print(f"  Relevance judgments: {len(collection.qrels)} queries with qrels")

# Sample query
sample_qid = list(collection.queries.keys())[0]
sample_query = collection.queries[sample_qid]
print(f"\nSample query {sample_qid}: {sample_query[:60]}...")

## Initialize SetBased Model

In [ ]:
model = SetBasedModel(collection)
print(f"Model: {model.get_model()}")
print(f"Vectorizer: IDF * termset frequency")
print(f"Weighting: Uniform (no graph-based node weights)")

## Fit Model (Mine Termsets)

This step runs Apriori on each query to find frequent termsets across documents.

In [ ]:
import time

# Fit model on all queries with minimum support of 1 document
print("Fitting model (Apriori termset mining)...")
start = time.time()

# Use a subset of queries for speed in demo (first 10)
demo_queries = list(collection.queries.items())[:10]
demo_query_dict = dict(demo_queries)

model.fit(queries=demo_query_dict, min_freq=1, stopwords=False)

elapsed = time.time() - start
print(f"\nFit complete in {elapsed:.2f}s")
print(f"Queries fitted: {len(model._query_ids)}")
print(f"Total termsets mined: {sum(len(ts) for ts in model._docVectors)}")

## Evaluate Rankings

In [ ]:
# Evaluate with metric cutoff at top 10
print("Evaluating...")
precisions, recalls = model.evaluate(k=10)

mean_prec = np.mean(precisions)
mean_rec = np.mean(recalls)

print(f"\nResults (first 10 queries, k=10):")
print(f"  Mean Precision: {mean_prec:.3f}")
print(f"  Mean Recall: {mean_rec:.3f}")
print(f"\nPer-query precision:")
for i, (qid, prec) in enumerate(zip(model._query_ids[:10], precisions[:10])):
    print(f"  Query {qid:3s}: {prec:.3f}")

## Inspect a Ranking

In [ ]:
# Show top 5 documents for first query
if model.ranking:
    first_ranking = model.ranking[0]
    first_qid = model._query_ids[0]
    
    print(f"Top 5 documents for query {first_qid}:")
    print(f"Query: {collection.queries[first_qid][:70]}...\n")
    
    relevant = collection.relevant[first_qid]
    for rank, doc_id in enumerate(first_ranking[:5], start=1):
        is_rel = "✓ RELEVANT" if doc_id in relevant else "  -"
        print(f"  {rank}. Doc {doc_id:4s} {is_rel}")

## Summary

SetBased model successfully:
- Loads the CF collection (1,239 docs, 100 queries)
- Mines frequent termsets using Apriori per query
- Ranks documents by cosine similarity with IDF weighting
- Evaluates precision and recall at a given cutoff (k=10)

This is the foundation for graph-based variants (GSB, windowed GSB, spectral models) that add graph-derived node weighting.